# 04 — Negotiation & Coordination
**Agentic X | Studio 67** — Real components only. Run from the repository root.


## 1. Project Setup

Connect to the Agentic X repository and verify the required project files.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/ProjectsSpaceHub/AgenticX.git"
ROOT = Path("/content/AgenticX")

if not ROOT.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(ROOT)],
        check=True
    )
else:
    if not (ROOT / ".git").exists():
        raise RuntimeError("Project folder is not a Git repository")

    changes = subprocess.run(
        ["git", "-C", str(ROOT), "status", "--porcelain"],
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()

    if not changes:
        subprocess.run(
            ["git", "-C", str(ROOT), "pull", "--ff-only"],
            check=True
        )
    else:
        print("Local changes detected. Skipping pull.")

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

required_files = [
    "core/models.py",
    "core/environment.py",
    "core/communication.py",
    "core/negotiation.py",
    "agents/creative.py",
    "agents/finance.py",
    "agents/safety.py",
    "agents/operations.py"
]

for file in required_files:
    assert (ROOT / file).is_file(), f"Missing: {file}"

print("GitHub project connected successfully!")
print("All required files found!")
print("Project path:", ROOT)

GitHub project connected successfully!
All required files found!
Project path: /content/AgenticX


### 2. Dependencies & Agent Imports

Install dependencies and import the four agents and shared components.

In [2]:
import subprocess
import sys

# Install project dependencies
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-q", "-r", str(ROOT / "requirements.txt")
    ],
    check=True
)

# Shared components
from core.environment import load_dataset, get_case
from core.models import AgentEvaluation, ACLMessage, RunResult
from core.communication import MessageBus
from core.negotiation import NegotiationEngine

# Four independent agents
from agents.creative import CreativeAgent
from agents.finance import FinanceAgent
from agents.safety import SafetyAgent
from agents.operations import OperationsAgent

# Verify dataset
cases_df, options_df = load_dataset()

assert len(cases_df) == 150
assert len(options_df) == 450

print("All dependencies installed!")
print("All four agents imported successfully!")
print("Communication and negotiation modules ready!")
print("Dataset:", len(cases_df), "cases,", len(options_df), "options")

All dependencies installed!
All four agents imported successfully!
Communication and negotiation modules ready!
Dataset: 150 cases, 450 options


### 3. Gemini Configuration & Agent Initialization

Configure Gemini securely and initialize the four real agents.

In [33]:
import os
from getpass import getpass
from google.colab import userdata

# Load Gemini API key securely
try:
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass("Enter your Gemini API key: ")

if not api_key.strip():
    raise ValueError("Gemini API key is required.")

os.environ["GOOGLE_API_KEY"] = api_key.strip()
os.environ["AGENTICX_MODEL"] = "gemini-3.5-flash-lite"

del api_key

# Initialize real agents
creative_agent = CreativeAgent()
finance_agent = FinanceAgent()
safety_agent = SafetyAgent()
operations_agent = OperationsAgent()

agents = [
    creative_agent,
    finance_agent,
    safety_agent,
    operations_agent
]

assert len(agents) == 4
assert len({agent.name for agent in agents}) == 4

print("Gemini configured successfully!")
print("All four agents initialized!")

for agent in agents:
    print("✓", agent.name)

Enter your Gemini API key: ··········
Gemini configured successfully!
All four agents initialized!
✓ Creative
✓ Finance
✓ Safety
✓ Operations


### 4. Negotiation Rules & Scenario Selection

Define negotiation constraints and inspect the available solutions.

In [4]:
import pandas as pd
from IPython.display import display

# Negotiation settings
MAX_ROUNDS = 3
REQUIRE_ALL_APPROVALS = True

# Select a real scenario
case, options = get_case("SCR-001")

assert len(options) == 3
assert len({o["option_id"] for o in options}) == 3

print("Case:", case["case_id"])
print("Incident:", case["incident"])
print("Maximum negotiation rounds:", MAX_ROUNDS)
print("All four agents must approve:", REQUIRE_ALL_APPROVALS)

display(
    pd.DataFrame(options)[
        ["option_id", "option_name", "cost_sar", "duration_min"]
    ]
)

Case: SCR-001
Incident: Main LED screen failed during final rehearsal
Maximum negotiation rounds: 3
All four agents must approve: True


,option_id,option_name,cost_sar,duration_min
0,SCR-001-O1,Use backup LED,1290,40
1,SCR-001-O2,Rent replacement screen,5920,70
2,SCR-001-O3,Continue with projection,1210,25


### 5. Multi-Round Negotiation Engine

Upgrade the coordinator to handle agent conflicts, alternative proposals, and human escalation.

In [5]:
from pathlib import Path

# Locate the current negotiation engine
negotiation_path = ROOT / "core" / "negotiation.py"

assert negotiation_path.exists(), "Negotiation file not found"

# Read the existing implementation
original_code = negotiation_path.read_text(
    encoding="utf-8"
)

assert "class NegotiationEngine" in original_code
assert "def run(" in original_code

# Save a backup outside the repository
backup_path = Path("/content/negotiation_original.py")

if not backup_path.exists():
    backup_path.write_text(
        original_code,
        encoding="utf-8"
    )

print("Current negotiation engine verified!")
print("Original implementation backed up!")
print("Backup path:", backup_path)
print("Ready to upgrade the negotiation engine.")

Current negotiation engine verified!
Original implementation backed up!
Backup path: /content/negotiation_original.py
Ready to upgrade the negotiation engine.


### 5.1. Adaptive Proposal Selection

Prioritize alternative solutions based on agent feedback and negotiation constraints.

In [6]:
import math

negotiation_source = '''
"""Adaptive multi-agent negotiation for Studio 67."""

import math
from core.communication import MessageBus
from core.models import RunResult


class NegotiationEngine:

    def __init__(
        self, creative, finance, safety,
        operations, max_rounds=3
    ):
        self.agents = [
            creative, finance, safety, operations
        ]

        if not isinstance(max_rounds, int) or max_rounds < 1:
            raise ValueError("max_rounds must be positive")

        self.max_rounds = max_rounds

    @staticmethod
    def _number(value, default=0.0):
        try:
            number = float(value)
            return number if math.isfinite(number) else default
        except (TypeError, ValueError):
            return default

    @staticmethod
    def _is_true(value):
        return (
            value is True
            or value == 1
            or (
                isinstance(value, str)
                and value.strip().lower() == "true"
            )
        )

    def _priority(self, option, objections):
        """Rank options using previous agent objections."""

        cost = self._number(option.get("cost_sar"))
        duration = self._number(option.get("duration_min"))
        quality = self._number(option.get("quality_score"))
        savings = self._number(
            option.get("gross_savings_sar")
        )

        safety_ready = all([
            self._is_true(option.get("safety_ok")),
            self._is_true(option.get("permit_ok")),
            self._is_true(
                option.get("mitigation_verified")
            )
        ])

        risk = self._number(
            option.get("risk_score_after_mitigation"),
            default=999
        )

        if "Safety" in objections:
            return (
                int(safety_ready),
                -risk,
                savings - cost,
                quality,
                -duration
            )

        if "Finance" in objections:
            return (
                savings - cost,
                quality,
                -duration,
                int(safety_ready),
                -risk
            )

        if "Operations" in objections:
            return (
                -duration,
                quality,
                savings - cost,
                int(safety_ready),
                -risk
            )

        return (
            quality,
            -duration,
            savings - cost,
            int(safety_ready),
            -risk
        )
'''

# Validate syntax before modifying the actual Python file
compile(
    negotiation_source,
    "negotiation_stage_1",
    "exec"
)

print("Adaptive proposal selection code validated!")
print("Original negotiation.py remains unchanged.")
print("Ready for the multi-round negotiation logic.")

Adaptive proposal selection code validated!
Original negotiation.py remains unchanged.
Ready for the multi-round negotiation logic.


### 5.2. Agent Evaluation & ACL Responses

Collect real agent evaluations and record their decisions using ACL messages.

In [7]:
# Add the shared evaluation model
negotiation_source = negotiation_source.replace(
    "from core.models import RunResult",
    "from core.models import RunResult, AgentEvaluation"
)

# Add the agent evaluation method
negotiation_source += '''
    def _evaluate_proposal(
        self, case, option, round_number, bus
    ):
        """Collect real evaluations from all four agents."""

        evaluations = []

        for agent in self.agents:

            # Coordinator requests an evaluation
            request = bus.send(
                round_number,
                "Coordinator",
                agent.name,
                "REQUEST",
                f"Evaluate option {option['option_id']}"
            )

            # Real agent evaluation
            evaluation = agent.evaluate(case, option)

            # Verify the returned evaluation
            if not isinstance(evaluation, AgentEvaluation):
                raise TypeError(
                    f"{agent.name} returned an invalid evaluation"
                )

            if evaluation.agent != agent.name:
                raise ValueError("Agent name mismatch")

            if evaluation.option_id != option["option_id"]:
                raise ValueError("Option ID mismatch")

            evaluations.append(evaluation)

            # Share evaluation reasoning
            bus.send(
                round_number,
                agent.name,
                "Coordinator",
                "INFORM",
                evaluation.reason,
                in_reply_to=request.message_id
            )

            # Record approval or refusal
            performative = (
                "ACCEPT" if evaluation.approved else "REFUSE"
            )

            bus.send(
                round_number,
                agent.name,
                "ALL",
                performative,
                f"{option['option_id']}: {evaluation.reason}"
            )

        return evaluations
'''

# Validate Python syntax
compile(
    negotiation_source,
    "negotiation_stage_2",
    "exec"
)

print("Agent evaluation logic validated!")
print("All four agents included!")
print("ACL responses configured!")
print("Original negotiation.py remains unchanged.")

Agent evaluation logic validated!
All four agents included!
ACL responses configured!
Original negotiation.py remains unchanged.


### 5.3. Multi-Round Negotiation & Conflict Resolution

Evaluate one proposal per round, adapt to agent objections, and escalate unresolved conflicts.

In [8]:
negotiation_source += '''
    def run(self, case, options):
        """Run adaptive negotiation across multiple rounds."""

        bus = MessageBus(case["case_id"])
        remaining = list(options)
        all_evaluations = []
        objections = set()
        rounds_used = 0

        # Prevent duplicate proposals
        option_ids = [o["option_id"] for o in remaining]
        if len(option_ids) != len(set(option_ids)):
            raise ValueError("Duplicate option IDs")

        for round_number in range(1, self.max_rounds + 1):

            if not remaining:
                break

            rounds_used = round_number

            # Announce a new negotiation round
            bus.send(
                round_number,
                "Coordinator",
                "ALL",
                "CFP",
                f"Request proposals for round {round_number}"
            )

            # Select an alternative based on previous objections
            proposal = max(
                remaining,
                key=lambda o: self._priority(o, objections)
            )

            remaining.remove(proposal)

            bus.send(
                round_number,
                "Coordinator",
                "ALL",
                "PROPOSE",
                f"Proposed option: {proposal['option_id']}"
            )

            # Collect real decisions from all four agents
            evaluations = self._evaluate_proposal(
                case, proposal, round_number, bus
            )

            all_evaluations.extend(evaluations)

            rejected = [
                e for e in evaluations
                if not e.approved
            ]

            # Agreement requires approval from all four agents
            if not rejected:

                bus.send(
                    round_number,
                    "Coordinator",
                    "ALL",
                    "ACCEPT",
                    f"Agreement reached: {proposal['option_id']}"
                )

                return RunResult(
                    case_id=case["case_id"],
                    status="AGREED",
                    selected_option_id=proposal["option_id"],
                    rounds=round_number,
                    evaluations=all_evaluations,
                    messages=bus.messages,
                    reason="All four agents approved the solution"
                )

            # Learn which agents objected
            objections = {
                evaluation.agent
                for evaluation in rejected
            }

            bus.send(
                round_number,
                "Coordinator",
                "ALL",
                "INFORM",
                "Proposal rejected by: "
                + ", ".join(sorted(objections))
            )

        # No acceptable proposal was found
        bus.send(
            max(1, rounds_used),
            "Coordinator",
            "Human Manager",
            "REQUEST",
            "Human escalation: no solution received "
            "approval from all four agents."
        )

        return RunResult(
            case_id=case["case_id"],
            status="ESCALATED",
            selected_option_id=None,
            rounds=rounds_used,
            evaluations=all_evaluations,
            messages=bus.messages,
            reason="No acceptable solution within the negotiation limit"
        )
'''

# Check syntax without modifying negotiation.py
compile(
    negotiation_source,
    "negotiation_stage_3",
    "exec"
)

print("Multi-round negotiation logic validated!")
print("Adaptive alternative selection configured!")
print("Agreement and escalation rules configured!")
print("Ready for integration testing.")

Multi-round negotiation logic validated!
Adaptive alternative selection configured!
Agreement and escalation rules configured!
Ready for integration testing.


### 5.4. Negotiation Engine Validation

Load the upgraded engine and verify adaptive proposal selection.

In [9]:
# Load the upgraded engine temporarily
namespace = {}

exec(
    compile(negotiation_source, "negotiation_candidate", "exec"),
    namespace
)

CandidateNegotiationEngine = namespace["NegotiationEngine"]

candidate_engine = CandidateNegotiationEngine(
    creative_agent,
    finance_agent,
    safety_agent,
    operations_agent,
    max_rounds=MAX_ROUNDS
)

# Test proposal selection under different objections
for objections in [
    set(),
    {"Finance"},
    {"Safety"},
    {"Operations"}
]:
    selected = max(
        options,
        key=lambda o: candidate_engine._priority(
            o, objections
        )
    )

    print(
        "Objections:", sorted(objections) or "None",
        "| Selected:", selected["option_id"],
        "|", selected["option_name"]
    )

# Ensure the original file remains unchanged
assert negotiation_path.read_text(
    encoding="utf-8"
) == original_code

print("\nAdaptive selection test passed!")
print("Original negotiation.py is still unchanged.")

Objections: None | Selected: SCR-001-O1 | Use backup LED
Objections: ['Finance'] | Selected: SCR-001-O3 | Continue with projection
Objections: ['Safety'] | Selected: SCR-001-O1 | Use backup LED
Objections: ['Operations'] | Selected: SCR-001-O3 | Continue with projection

Adaptive selection test passed!
Original negotiation.py is still unchanged.


### 5.5. Real Agent Evaluation Test

Evaluate a real proposal using all four agents and verify ACL responses.

In [10]:
# Create a communication bus for the selected case
test_bus = MessageBus(case["case_id"])

# Select the first real proposal
test_proposal = options[0]

print("Testing proposal:", test_proposal["option_name"])
print("Collecting real agent evaluations...\n")

# Evaluate using all four real agents
test_evaluations = candidate_engine._evaluate_proposal(
    case,
    test_proposal,
    round_number=1,
    bus=test_bus
)

# Display agent decisions
evaluation_df = pd.DataFrame([
    evaluation.model_dump()
    for evaluation in test_evaluations
])

display(evaluation_df)

# Validate evaluations and communication
assert len(test_evaluations) == 4
assert len(test_bus.messages) == 12

assert {
    evaluation.agent
    for evaluation in test_evaluations
} == {
    "Creative", "Finance", "Safety", "Operations"
}

assert all(
    evaluation.option_id == test_proposal["option_id"]
    for evaluation in test_evaluations
)

print("\nReal agent evaluation completed!")
print("ACL messages:", len(test_bus.messages))
print("All evaluation checks passed!")

Testing proposal: Use backup LED



,agent,option_id,approved,score,reason,blocking
0,Creative,SCR-001-O1,True,89.0,The solution replaces the failed screen using ...,False
1,Finance,SCR-001-O1,True,100.0,Committed plus option cost: 1290.00 SAR versus...,True
2,Safety,SCR-001-O1,True,100.0,All safety checks passed,True
3,Operations,SCR-001-O1,True,95.0,The option is highly feasible operationally. E...,True



Real agent evaluation completed!
ACL messages: 12
All evaluation checks passed!


### 5.6. Direct Agreement Test

Execute a real negotiation and verify the final decision and ACL messages.

In [11]:
# Execute the complete negotiation process
direct_result = candidate_engine.run(case, options)

print("Case:", direct_result.case_id)
print("Status:", direct_result.status)
print("Selected Option:", direct_result.selected_option_id)
print("Rounds:", direct_result.rounds)
print("Evaluations:", len(direct_result.evaluations))
print("ACL Messages:", len(direct_result.messages))

# Validate the result
assert direct_result.is_mock is False
assert 1 <= direct_result.rounds <= MAX_ROUNDS

if direct_result.status == "AGREED":
    selected_evaluations = [
        e for e in direct_result.evaluations
        if e.option_id == direct_result.selected_option_id
    ]

    assert len(selected_evaluations) == 4
    assert all(e.approved for e in selected_evaluations)

    print("\nAgreement verified successfully!")

else:
    print("\nNo agreement reached. Review agent objections.")

# Display actual ACL communication
messages_df = pd.DataFrame([
    message.model_dump()
    for message in direct_result.messages
])

display(
    messages_df[
        ["round", "sender", "receiver", "performative", "content"]
    ]
)

print("\nFull negotiation execution completed!")

Case: SCR-001
Status: AGREED
Selected Option: SCR-001-O1
Rounds: 1
Evaluations: 4
ACL Messages: 15

Agreement verified successfully!


,round,sender,receiver,performative,content
0,1,Coordinator,ALL,CFP,Request proposals for round 1
1,1,Coordinator,ALL,PROPOSE,Proposed option: SCR-001-O1
2,1,Coordinator,Creative,REQUEST,Evaluate option SCR-001-O1
3,1,Creative,Coordinator,INFORM,The solution directly resolves the main LED sc...
4,1,Creative,ALL,ACCEPT,SCR-001-O1: The solution directly resolves the...
5,1,Coordinator,Finance,REQUEST,Evaluate option SCR-001-O1
6,1,Finance,Coordinator,INFORM,Committed plus option cost: 1290.00 SAR versus...
7,1,Finance,ALL,ACCEPT,SCR-001-O1: Committed plus option cost: 1290.0...
8,1,Coordinator,Safety,REQUEST,Evaluate option SCR-001-O1
9,1,Safety,Coordinator,INFORM,All safety checks passed



Full negotiation execution completed!


### 6. Conflict Scenario Preparation

Inspect real agent objections and identify suitable scenarios for multi-round negotiation.

In [12]:
# Preview real conflicts without calling Gemini
conflict_preview = []

for case_id in ["BDG-001", "SFT-001"]:

    test_case, test_options = get_case(case_id)

    ranked_options = sorted(
        test_options,
        key=lambda o: candidate_engine._priority(o, set()),
        reverse=True
    )

    for rank, option in enumerate(ranked_options, start=1):

        finance = finance_agent.evaluate(test_case, option)
        safety = safety_agent.evaluate(test_case, option)

        conflict_preview.append({
            "case_id": case_id,
            "proposal_order": rank,
            "option_id": option["option_id"],
            "finance_approved": finance.approved,
            "safety_approved": safety.approved,
            "objections": ", ".join([
                agent.agent
                for agent in [finance, safety]
                if not agent.approved
            ]) or "None"
        })

conflict_df = pd.DataFrame(conflict_preview)

display(conflict_df)

assert len(conflict_df) == 6

print("Conflict scenario preview completed!")

,case_id,proposal_order,option_id,finance_approved,safety_approved,objections
0,BDG-001,1,BDG-001-O3,False,True,Finance
1,BDG-001,2,BDG-001-O2,True,True,None
2,BDG-001,3,BDG-001-O1,True,True,None
3,SFT-001,1,SFT-001-O3,True,True,None
4,SFT-001,2,SFT-001-O2,True,True,None
5,SFT-001,3,SFT-001-O1,True,False,Safety


Conflict scenario preview completed!


### 6.1. Multi-Round Conflict Resolution Test

Test real negotiation after a financial objection.

In [13]:
# Run a real budget conflict scenario
budget_case, budget_options = get_case("BDG-001")

budget_result = candidate_engine.run(
    budget_case,
    budget_options
)

print("Case:", budget_result.case_id)
print("Status:", budget_result.status)
print("Selected Option:", budget_result.selected_option_id)
print("Rounds:", budget_result.rounds)

# Show agent decisions across negotiation rounds
budget_evaluations_df = pd.DataFrame([
    e.model_dump()
    for e in budget_result.evaluations
])

display(
    budget_evaluations_df[
        ["option_id", "agent", "approved", "score", "reason"]
    ]
)

# Verify the real Finance objection
first_option = budget_evaluations_df["option_id"].iloc[0]

assert first_option == "BDG-001-O3"

assert (
    (budget_evaluations_df["option_id"] == first_option)
    & (budget_evaluations_df["agent"] == "Finance")
    & (~budget_evaluations_df["approved"])
).any()

assert budget_result.rounds >= 2

print("\nFinancial conflict detected successfully!")
print("Negotiation continued beyond round 1!")

Case: BDG-001
Status: AGREED
Selected Option: BDG-001-O2
Rounds: 2


,option_id,agent,approved,score,reason
0,BDG-001-O3,Creative,True,99.0,"The solution keeps the premium package, delive..."
1,BDG-001-O3,Finance,False,0.0,Projected spend after action: 29176.00 SAR ver...
2,BDG-001-O3,Safety,True,100.0,All safety checks passed
3,BDG-001-O3,Operations,True,92.0,Operationally feasible: execution time of 20 m...
4,BDG-001-O2,Creative,True,82.0,The standard AV package maintains strong event...
5,BDG-001-O2,Finance,True,100.0,Projected spend after action: 20686.00 SAR ver...
6,BDG-001-O2,Safety,True,100.0,All safety checks passed
7,BDG-001-O2,Operations,True,95.0,Operationally feasible: execution time of 50 m...



Financial conflict detected successfully!
Negotiation continued beyond round 1!


### 6.2. Human Escalation Test

Verify that unresolved conflicts are escalated after three negotiation rounds.

In [14]:
# Load the safety conflict scenario
safety_case, safety_options = get_case("SFT-001")

# Controlled stress test: no execution time remains
escalation_case = safety_case.copy()
escalation_case["time_left_min"] = 0

# Run real negotiation
escalation_result = candidate_engine.run(
    escalation_case,
    safety_options
)

print("Case:", escalation_result.case_id)
print("Status:", escalation_result.status)
print("Rounds:", escalation_result.rounds)
print("Selected Option:", escalation_result.selected_option_id)
print("Reason:", escalation_result.reason)

# Validate escalation
assert escalation_result.status == "ESCALATED"
assert escalation_result.selected_option_id is None
assert escalation_result.rounds == 3
assert escalation_result.is_mock is False

# Verify human escalation message
human_messages = [
    message
    for message in escalation_result.messages
    if message.receiver == "Human Manager"
]

assert len(human_messages) == 1
assert human_messages[0].performative == "REQUEST"

# Display agent refusals
refusals = [
    message.model_dump()
    for message in escalation_result.messages
    if message.performative == "REFUSE"
]

display(pd.DataFrame(refusals)[
    ["round", "sender", "content"]
])

print("\nHuman escalation test passed!")

Case: SFT-001
Status: ESCALATED
Rounds: 3
Selected Option: None
Reason: No acceptable solution within the negotiation limit


,round,sender,content
0,1,Operations,SFT-001-O3: Operational constraints failed: time
1,2,Creative,SFT-001-O1: The proposed option leaves the eme...
2,2,Safety,"SFT-001-O1: Safety veto: safety_ok, permit_ok,..."
3,2,Operations,SFT-001-O1: Operational constraints failed: time
4,3,Operations,SFT-001-O2: Operational constraints failed: time



Human escalation test passed!


### 7. Final Validation & Engine Integration

Validate negotiation outcomes and save the upgraded Python engine.

In [15]:
import importlib
import inspect

# Verify all three negotiation outcomes
assert direct_result.status == "AGREED"
assert direct_result.rounds == 1

assert budget_result.status == "AGREED"
assert budget_result.rounds == 2

assert escalation_result.status == "ESCALATED"
assert escalation_result.rounds == 3
assert escalation_result.selected_option_id is None

# Verify agreement and message integrity
for result in [
    direct_result,
    budget_result,
    escalation_result
]:
    assert result.is_mock is False

    message_ids = [
        message.message_id
        for message in result.messages
    ]

    assert len(message_ids) == len(set(message_ids))

    if result.status == "AGREED":
        selected = [
            evaluation
            for evaluation in result.evaluations
            if evaluation.option_id == result.selected_option_id
        ]

        assert len(selected) == 4
        assert all(e.approved for e in selected)

print("All negotiation scenarios validated!")

# Validate Python source
compile(
    negotiation_source,
    str(negotiation_path),
    "exec"
)

# Save upgraded engine locally
negotiation_path.write_text(
    negotiation_source,
    encoding="utf-8"
)

# Reload the actual Python module
import core.negotiation as negotiation_module

importlib.reload(negotiation_module)

NegotiationEngine = negotiation_module.NegotiationEngine

# Verify the upgraded module
assert Path(
    inspect.getfile(NegotiationEngine)
).resolve() == negotiation_path.resolve()

assert hasattr(NegotiationEngine, "_priority")
assert hasattr(NegotiationEngine, "_evaluate_proposal")

print("Upgraded negotiation.py saved successfully!")
print("Python module reloaded successfully!")
print("File:", negotiation_path)

All negotiation scenarios validated!
Upgraded negotiation.py saved successfully!
Python module reloaded successfully!
File: /content/AgenticX/core/negotiation.py


### 8. System Integration Check

Verify that the upgraded negotiation engine integrates with the project runner.

In [16]:
import importlib
import core.runner as runner_module
import core.negotiation as negotiation_module

# Reload the project runner
importlib.reload(runner_module)

# Build the integrated engine
integrated_engine = runner_module.build_engine()

# Verify integration
assert isinstance(
    integrated_engine,
    negotiation_module.NegotiationEngine
)

assert len(integrated_engine.agents) == 4
assert integrated_engine.max_rounds == 3

assert {
    agent.name for agent in integrated_engine.agents
} == {
    "Creative", "Finance",
    "Safety", "Operations"
}

assert negotiation_path.read_text(
    encoding="utf-8"
) == negotiation_source

print("Project runner integration: PASS")
print("Four real agents connected: PASS")
print("Upgraded negotiation engine: PASS")
print("Notebook 05 compatibility: PASS")

Project runner integration: PASS
Four real agents connected: PASS
Upgraded negotiation engine: PASS
Notebook 05 compatibility: PASS


### 9. Export Negotiation Engine

Download the upgraded negotiation module for GitHub integration.

In [17]:
from google.colab import files

# Verify the final Python file
assert negotiation_path.exists()

assert negotiation_path.read_text(
    encoding="utf-8"
) == negotiation_source

print("Final negotiation engine verified!")
print("Downloading negotiation.py...")

files.download(str(negotiation_path))

Final negotiation engine verified!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 10. Time-Aware Negotiation

Track remaining time to prevent decisions that exceed the event deadline.

In [18]:
import math

def remaining_time_minutes(initial_minutes, elapsed_seconds):
    """Calculate remaining event time after negotiation delay."""

    try:
        initial = float(initial_minutes)
        elapsed = float(elapsed_seconds)
    except (TypeError, ValueError):
        raise ValueError("Invalid time values")

    if (
        not math.isfinite(initial)
        or not math.isfinite(elapsed)
        or initial < 0
        or elapsed < 0
    ):
        raise ValueError("Time values must be finite and non-negative")

    return max(0.0, initial - elapsed / 60)


# Test 1: 30 minutes available, 10 minutes spent negotiating
remaining = remaining_time_minutes(30, 600)

assert remaining == 20
assert 25 > remaining

print("Test 1: Insufficient time correctly detected")


# Test 2: 30 minutes available, 1 minute spent negotiating
remaining = remaining_time_minutes(30, 60)

assert remaining == 29
assert 25 <= remaining

print("Test 2: Feasible execution time verified")


# Test 3: Remaining time cannot become negative
assert remaining_time_minutes(5, 600) == 0

print("Test 3: Time cannot become negative")
print("Time calculation tests passed!")

Test 1: Insufficient time correctly detected
Test 2: Feasible execution time verified
Test 3: Time cannot become negative
Time calculation tests passed!


### 10.1. Real-Time Negotiation Tracking

Update the available event time as negotiation progresses.

In [19]:
# Add real-time tracking to the negotiation engine

updated_source = negotiation_source

old_import = "import math\n"
old_start = '        bus = MessageBus(case["case_id"])\n'

old_evaluation = '''            evaluations = self._evaluate_proposal(
                case, proposal, round_number, bus
            )'''

# Ensure we are modifying the correct code
assert "import time\n" not in updated_source
assert updated_source.count(old_start) == 1
assert updated_source.count(old_evaluation) == 1

# Add timing library
updated_source = updated_source.replace(
    old_import,
    "import math\nimport time\n",
    1
)

# Start measuring negotiation time
updated_source = updated_source.replace(
    old_start,
    '''        bus = MessageBus(case["case_id"])
        started_at = time.perf_counter()
        initial_minutes = self._number(
            case.get("time_left_min")
        )
''',
    1
)

# Update remaining time before evaluating each proposal
updated_source = updated_source.replace(
    old_evaluation,
    '''            current_case = dict(case)

            elapsed_minutes = (
                time.perf_counter() - started_at
            ) / 60.0

            current_case["time_left_min"] = max(
                0.0,
                initial_minutes - elapsed_minutes
            )

            evaluations = self._evaluate_proposal(
                current_case, proposal, round_number, bus
            )''',
    1
)

# Verify code syntax
compile(
    updated_source,
    "time_aware_negotiation",
    "exec"
)

negotiation_source = updated_source

print("Real-time tracking added successfully!")
print("Remaining time updates before each proposal evaluation.")
print("Python syntax validation passed!")

Real-time tracking added successfully!
Remaining time updates before each proposal evaluation.
Python syntax validation passed!


### 10.2. Final Deadline Validation

Recheck the remaining time before confirming an agreement.

In [20]:
# Add final deadline validation before agreement

old_agreement = '''            # Agreement requires approval from all four agents
            if not rejected:
'''

new_agreement = '''            # Recheck the deadline after agent evaluations
            elapsed_seconds = time.perf_counter() - started_at

            remaining_minutes = max(
                0.0,
                initial_minutes - elapsed_seconds / 60.0
            )

            duration = self._number(
                proposal.get("duration_min"),
                default=float("inf")
            )

            deadline_ok = duration <= remaining_minutes

            if not deadline_ok:
                bus.send(
                    round_number,
                    "Coordinator",
                    "ALL",
                    "REFUSE",
                    f"Deadline constraint failed for "
                    f"{proposal['option_id']}: "
                    f"{duration:.2f} minutes required, "
                    f"{remaining_minutes:.2f} available."
                )

            # Agreement requires approval and deadline compliance
            if not rejected and deadline_ok:
'''

old_objections = '''            objections = {
                evaluation.agent
                for evaluation in rejected
            }
'''

new_objections = '''            objections = {
                evaluation.agent
                for evaluation in rejected
            }

            if not deadline_ok:
                objections.add("Operations")
'''

# Ensure the expected sections exist exactly once
assert negotiation_source.count(old_agreement) == 1
assert negotiation_source.count(old_objections) == 1

updated_source = negotiation_source.replace(
    old_agreement,
    new_agreement,
    1
)

updated_source = updated_source.replace(
    old_objections,
    new_objections,
    1
)

# Validate syntax
compile(
    updated_source,
    "deadline_aware_negotiation",
    "exec"
)

negotiation_source = updated_source

print("Final deadline validation added!")
print("Negotiation time included in feasibility checks!")
print("Deadline violations prevent agreement!")
print("Python syntax validation passed!")

Final deadline validation added!
Negotiation time included in feasibility checks!
Deadline violations prevent agreement!
Python syntax validation passed!


### 10.3. Time-Aware Engine Testing

Test the updated negotiation engine using real agents and measure execution time.

In [21]:
import time

# Load the updated engine temporarily
time_namespace = {}

exec(
    compile(negotiation_source, "time_aware_engine", "exec"),
    time_namespace
)

TimeAwareEngine = time_namespace["NegotiationEngine"]

time_engine = TimeAwareEngine(
    creative_agent,
    finance_agent,
    safety_agent,
    operations_agent,
    max_rounds=MAX_ROUNDS
)

# Run a real scenario
time_case, time_options = get_case("SCR-001")

start = time.perf_counter()

time_result = time_engine.run(
    time_case,
    time_options
)

elapsed_seconds = time.perf_counter() - start

remaining_minutes = remaining_time_minutes(
    time_case["time_left_min"],
    elapsed_seconds
)

print("Status:", time_result.status)
print("Selected Option:", time_result.selected_option_id)
print("Rounds:", time_result.rounds)
print("Negotiation Time:", round(elapsed_seconds, 2), "seconds")
print("Remaining Time:", round(remaining_minutes, 2), "minutes")

assert time_result.status in {"AGREED", "ESCALATED"}
assert time_result.is_mock is False
assert len(time_result.evaluations) >= 4

print("\nTime-aware negotiation test passed!")

Status: AGREED
Selected Option: SCR-001-O1
Rounds: 1
Negotiation Time: 295.23 seconds
Remaining Time: 95.08 minutes

Time-aware negotiation test passed!


### 10.4. Deadline Constraint Test

Verify that negotiation delays can make a solution infeasible.

In [22]:
# Use the actual execution time from the previous test
selected_option = next(
    o for o in time_options
    if o["option_id"] == time_result.selected_option_id
)

duration = float(selected_option["duration_min"])

# Set a tight deadline that allows the solution initially,
# but not after the measured negotiation delay.
tight_time = duration + elapsed_seconds / 120

remaining = remaining_time_minutes(
    tight_time,
    elapsed_seconds
)

assert duration <= tight_time
assert duration > remaining

print("Solution duration:", duration, "minutes")
print("Initial time:", round(tight_time, 2), "minutes")
print("Time after negotiation:", round(remaining, 2), "minutes")
print("Deadline decision: REJECT")

print("\nDeadline constraint calculation test passed!")

Solution duration: 40.0 minutes
Initial time: 42.46 minutes
Time after negotiation: 37.54 minutes
Deadline decision: REJECT

Deadline constraint calculation test passed!


### 10.5. Engine Deadline Validation

Verify deadline decisions using the negotiation engine's own logic.

In [23]:
# Add a reusable deadline check to the engine

deadline_method = '''
    @staticmethod
    def _deadline_check(duration, initial_minutes, elapsed_seconds):
        """Check execution feasibility after negotiation time."""

        try:
            duration = float(duration)
            initial_minutes = float(initial_minutes)
            elapsed_seconds = float(elapsed_seconds)
        except (TypeError, ValueError, OverflowError):
            return False, 0.0, float("inf")

        values = [duration, initial_minutes, elapsed_seconds]

        if not all(math.isfinite(v) and v >= 0 for v in values):
            return False, 0.0, duration

        remaining = max(
            0.0,
            initial_minutes - elapsed_seconds / 60.0
        )

        return duration <= remaining, remaining, duration

'''

old_check = '''            elapsed_seconds = time.perf_counter() - started_at

            remaining_minutes = max(
                0.0,
                initial_minutes - elapsed_seconds / 60.0
            )

            duration = self._number(
                proposal.get("duration_min"),
                default=float("inf")
            )

            deadline_ok = duration <= remaining_minutes'''

new_check = '''            deadline_ok, remaining_minutes, duration = (
                self._deadline_check(
                    proposal.get("duration_min"),
                    initial_minutes,
                    time.perf_counter() - started_at
                )
            )'''

assert negotiation_source.count(old_check) == 1
assert negotiation_source.count("    def run(self, case, options):") == 1

negotiation_source = negotiation_source.replace(
    "    def run(self, case, options):",
    deadline_method + "    def run(self, case, options):",
    1
).replace(old_check, new_check, 1)

# Load updated engine temporarily
namespace = {}

exec(
    compile(negotiation_source, "deadline_engine", "exec"),
    namespace
)

UpdatedEngine = namespace["NegotiationEngine"]

# Test using actual measured negotiation time
approved, remaining, required = UpdatedEngine._deadline_check(
    duration,
    tight_time,
    elapsed_seconds
)

assert approved is False
assert required > remaining

# Test a deadline with sufficient time
approved_normal, _, _ = UpdatedEngine._deadline_check(
    duration,
    100,
    elapsed_seconds
)

assert approved_normal is True

print("Tight deadline correctly rejected!")
print("Sufficient deadline correctly approved!")
print("Engine deadline validation passed!")

Tight deadline correctly rejected!
Sufficient deadline correctly approved!
Engine deadline validation passed!


### 11. Agent Failure Handling

Handle agent failures safely without generating artificial evaluations.

In [24]:
# Handle unexpected agent evaluation errors safely

old_evaluation = '''            evaluations = self._evaluate_proposal(
                current_case, proposal, round_number, bus
            )'''

new_evaluation = '''            try:
                evaluations = self._evaluate_proposal(
                    current_case, proposal, round_number, bus
                )

            except Exception as exc:
                error_type = type(exc).__name__

                bus.send(
                    round_number,
                    "Coordinator",
                    "ALL",
                    "INFORM",
                    f"Agent evaluation failed: {error_type}"
                )

                bus.send(
                    round_number,
                    "Coordinator",
                    "Human Manager",
                    "REQUEST",
                    "Human review required after agent failure"
                )

                return RunResult(
                    case_id=case["case_id"],
                    status="FAILED",
                    selected_option_id=None,
                    rounds=round_number,
                    evaluations=all_evaluations,
                    messages=bus.messages,
                    reason=f"Agent evaluation error: {error_type}"
                )'''

assert negotiation_source.count(old_evaluation) == 1

negotiation_source = negotiation_source.replace(
    old_evaluation,
    new_evaluation,
    1
)

compile(
    negotiation_source,
    "failure_safe_negotiation",
    "exec"
)

print("Agent failure handling added!")
print("Failed evaluations cannot produce agreement!")
print("Human review request configured!")
print("Python syntax validation passed!")

Agent failure handling added!
Failed evaluations cannot produce agreement!
Human review request configured!
Python syntax validation passed!


### 11.1. Agent Failure Recovery Test

Verify safe failure handling and human escalation when an agent encounters invalid data.

In [25]:
# Load the updated engine temporarily
failure_namespace = {}

exec(
    compile(negotiation_source, "failure_test_engine", "exec"),
    failure_namespace
)

FailureSafeEngine = failure_namespace["NegotiationEngine"]

failure_engine = FailureSafeEngine(
    creative_agent,
    finance_agent,
    safety_agent,
    operations_agent,
    max_rounds=3
)

# Run Finance first to test its failure without Gemini calls
failure_engine.agents = [
    finance_agent,
    creative_agent,
    safety_agent,
    operations_agent
]

# Load a real scenario
failure_case, failure_options = get_case("BDG-001")

# Controlled fault injection: invalid financial input
failure_case["projected_total_spend_sar"] = "invalid"

failure_result = failure_engine.run(
    failure_case,
    failure_options
)

print("Status:", failure_result.status)
print("Rounds:", failure_result.rounds)
print("Selected Option:", failure_result.selected_option_id)
print("Reason:", failure_result.reason)

# Validate safe failure handling
assert failure_result.status == "FAILED"
assert failure_result.selected_option_id is None
assert failure_result.rounds == 1
assert failure_result.is_mock is False

# Verify human review request
human_requests = [
    m for m in failure_result.messages
    if m.receiver == "Human Manager"
    and m.performative == "REQUEST"
]

assert len(human_requests) == 1

print("\nAgent failure handled successfully!")
print("Human review request verified!")
print("Agent failure recovery test passed!")

Status: FAILED
Rounds: 1
Selected Option: None
Reason: Agent evaluation error: ValueError

Agent failure handled successfully!
Human review request verified!
Agent failure recovery test passed!


### 11.2. Negotiation Robustness Tests

Validate duplicate detection, empty proposals, and consistent option ranking.

In [26]:
# Test 1: Reject duplicate proposals
try:
    failure_engine.run(case, [options[0], options[0]])
    raise AssertionError("Duplicate options were accepted")
except ValueError:
    print("Test 1: Duplicate proposals rejected!")


# Test 2: Handle an empty proposal list
empty_result = failure_engine.run(case, [])

assert empty_result.status == "ESCALATED"
assert empty_result.selected_option_id is None
assert empty_result.rounds == 0

print("Test 2: Empty proposals escalated safely!")


# Test 3: Verify consistent proposal ranking
rankings = []

for _ in range(5):
    ranking = [
        o["option_id"]
        for o in sorted(
            options,
            key=lambda o: failure_engine._priority(
                o, {"Finance"}
            ),
            reverse=True
        )
    ]
    rankings.append(ranking)

assert all(r == rankings[0] for r in rankings)

print("Test 3: Proposal ranking consistent across 5 runs!")


# Test 4: Reject invalid negotiation limits
try:
    FailureSafeEngine(
        creative_agent,
        finance_agent,
        safety_agent,
        operations_agent,
        max_rounds=0
    )
    raise AssertionError("Invalid round limit accepted")
except ValueError:
    print("Test 4: Invalid round limit rejected!")

print("\nAll robustness tests passed!")

Test 1: Duplicate proposals rejected!
Test 2: Empty proposals escalated safely!
Test 3: Proposal ranking consistent across 5 runs!
Test 4: Invalid round limit rejected!

All robustness tests passed!


### 12. Full Deadline Integration Test

Verify deadline enforcement during real multi-agent negotiation.

In [34]:
import time

# Load the latest negotiation engine
test_namespace = {}

exec(
    compile(
        negotiation_source,
        "final_test_engine",
        "exec"
    ),
    test_namespace
)

final_engine = test_namespace["NegotiationEngine"](
    creative_agent,
    finance_agent,
    safety_agent,
    operations_agent,
    max_rounds=MAX_ROUNDS
)

# Load a real scenario
deadline_case, deadline_options = get_case("SCR-001")

# Select one real solution
deadline_option = deadline_options[0]

# Controlled deadline test:
# Allow only 1.2 extra seconds beyond execution duration
deadline_case = deadline_case.copy()

deadline_case["time_left_min"] = (
    float(deadline_option["duration_min"]) + 0.02
)

print("Testing:", deadline_option["option_id"])
print("Execution duration:", deadline_option["duration_min"])
print("Initial time:", deadline_case["time_left_min"])
print("Running real negotiation...\n")

# Run the complete negotiation
start = time.perf_counter()

deadline_result = final_engine.run(
    deadline_case,
    [deadline_option]
)

elapsed = time.perf_counter() - start

# Find deadline refusal messages
deadline_refusals = [
    message
    for message in deadline_result.messages
    if message.sender == "Coordinator"
    and message.performative == "REFUSE"
    and "Deadline constraint failed" in message.content
]

# Display results
print("Status:", deadline_result.status)
print("Rounds:", deadline_result.rounds)
print("Elapsed seconds:", round(elapsed, 2))
print("Selected option:", deadline_result.selected_option_id)
print("Reason:", deadline_result.reason)

for message in deadline_refusals:
    print("Deadline refusal:", message.content)

# Validate the result
assert deadline_result.status == "ESCALATED", (
    f"Unexpected status: {deadline_result.reason}"
)

assert deadline_result.selected_option_id is None
assert deadline_result.rounds == 1
assert len(deadline_result.evaluations) == 4
assert len(deadline_refusals) == 1
assert deadline_result.is_mock is False

print("\nFull deadline integration test passed!")

Testing: SCR-001-O1
Execution duration: 40
Initial time: 40.02
Running real negotiation...



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Status: ESCALATED
Rounds: 1
Elapsed seconds: 1.55
Selected option: None
Reason: No acceptable solution within the negotiation limit
Deadline refusal: Deadline constraint failed for SCR-001-O1: 40.00 minutes required, 39.99 available.

Full deadline integration test passed!


### 13. Negotiation Parameters

Document the key variables affecting negotiation decisions.

In [35]:
parameters_df = pd.DataFrame([
    {
        "Variable": "max_rounds",
        "Purpose": "Maximum negotiation rounds",
        "Effect": "Limits negotiation attempts"
    },
    {
        "Variable": "time_left_min",
        "Purpose": "Available event time",
        "Effect": "Controls deadline feasibility"
    },
    {
        "Variable": "duration_min",
        "Purpose": "Solution execution duration",
        "Effect": "Prevents late solutions"
    },
    {
        "Variable": "cost_sar",
        "Purpose": "Solution cost",
        "Effect": "Influences financial decisions"
    },
    {
        "Variable": "objections",
        "Purpose": "Previous agent refusals",
        "Effect": "Guides alternative selection"
    },
    {
        "Variable": "elapsed_seconds",
        "Purpose": "Actual negotiation time",
        "Effect": "Reduces remaining event time"
    }
])

display(parameters_df)

assert len(parameters_df) == 6

print("Negotiation parameters documented successfully!")

,Variable,Purpose,Effect
0,max_rounds,Maximum negotiation rounds,Limits negotiation attempts
1,time_left_min,Available event time,Controls deadline feasibility
2,duration_min,Solution execution duration,Prevents late solutions
3,cost_sar,Solution cost,Influences financial decisions
4,objections,Previous agent refusals,Guides alternative selection
5,elapsed_seconds,Actual negotiation time,Reduces remaining event time


Negotiation parameters documented successfully!


### 14. Negotiation Scenario Documentation

Summarize initial conditions, agent interactions, and observed outcomes.

In [39]:
scenario_tests = [
    ("SCR-001", direct_result, "Direct agreement", "AGREED"),
    ("BDG-001", budget_result, "Financial conflict", "AGREED"),
    ("SFT-001", escalation_result, "Deadline conflict", "ESCALATED")
]

scenario_records = []

for case_id, result, test_type, expected in scenario_tests:

    scenario_case, _ = get_case(case_id)

    original_time = scenario_case["time_left_min"]

    test_time = (
        escalation_case["time_left_min"]
        if case_id == "SFT-001"
        else original_time
    )

    scenario_records.append({
        "Case ID": case_id,
        "Scenario": scenario_case["scenario"],
        "Test Type": test_type,
        "Original Time (min)": original_time,
        "Test Time (min)": test_time,
        "Agents": "Creative, Finance, Safety, Operations",
        "Expected": expected,
        "Observed": result.status,
        "Rounds": result.rounds,
        "Selected Option": result.selected_option_id,
        "ACL Messages": len(result.messages)
    })

scenario_summary_df = pd.DataFrame(scenario_records)

display(scenario_summary_df)

assert len(scenario_summary_df) == 3
assert (
    scenario_summary_df["Expected"]
    == scenario_summary_df["Observed"]
).all()

assert scenario_summary_df.loc[
    scenario_summary_df["Case ID"] == "SFT-001",
    "Original Time (min)"
].iloc[0] == 60

assert scenario_summary_df.loc[
    scenario_summary_df["Case ID"] == "SFT-001",
    "Test Time (min)"
].iloc[0] == 0

print("All three scenarios documented successfully!")

,Case ID,Scenario,Test Type,Original Time (min),Test Time (min),Agents,Expected,Observed,Rounds,Selected Option,ACL Messages
0,SCR-001,Screen Failure,Direct agreement,100,100,"Creative, Finance, Safety, Operations",AGREED,AGREED,1,SCR-001-O1,15
1,BDG-001,Budget Overrun,Financial conflict,80,80,"Creative, Finance, Safety, Operations",AGREED,AGREED,2,BDG-001-O2,30
2,SFT-001,Safety Conflict,Deadline conflict,60,0,"Creative, Finance, Safety, Operations",ESCALATED,ESCALATED,3,None,46


All three scenarios documented successfully!


### 14.1. Scenario Walkthroughs

Document scenario events, agent interactions, and observed decisions.

In [38]:
walkthroughs = []

for case_id, result, test_type, expected in scenario_tests:

    scenario_case, _ = get_case(case_id)

    # Original dataset time
    original_time = scenario_case["time_left_min"]

    # Actual time used during the test
    test_time = (
        escalation_case["time_left_min"]
        if case_id == "SFT-001"
        else original_time
    )

    refusals = [
        f"{e.agent} rejected {e.option_id}"
        for e in result.evaluations
        if not e.approved
    ]

    walkthroughs.append({
        "Case": case_id,
        "Event": scenario_case["incident"],
        "Original Time (min)": original_time,
        "Test Time (min)": test_time,
        "Agent Interactions": (
            "; ".join(refusals)
            if refusals else "All agents approved"
        ),
        "Expected Outcome": expected,
        "Observed Outcome": result.status,
        "Final Decision": (
            result.selected_option_id or "Human escalation"
        )
    })

walkthrough_df = pd.DataFrame(walkthroughs)

display(walkthrough_df)

assert len(walkthrough_df) == 3

assert (
    walkthrough_df["Expected Outcome"]
    == walkthrough_df["Observed Outcome"]
).all()

assert walkthrough_df.loc[
    walkthrough_df["Case"] == "SFT-001",
    "Original Time (min)"
].iloc[0] == 60

assert walkthrough_df.loc[
    walkthrough_df["Case"] == "SFT-001",
    "Test Time (min)"
].iloc[0] == 0

print("Scenario walkthroughs validated successfully!")

,Case,Event,Original Time (min),Test Time (min),Agent Interactions,Expected Outcome,Observed Outcome,Final Decision
0,SCR-001,Main LED screen failed during final rehearsal,100,100,All agents approved,AGREED,AGREED,SCR-001-O1
1,BDG-001,Supplier costs may exceed the event spending plan,80,80,Finance rejected BDG-001-O3,AGREED,AGREED,BDG-001-O2
2,SFT-001,Emergency exit partially blocked by stage setup,60,0,Operations rejected SFT-001-O3; Creative rejec...,ESCALATED,ESCALATED,Human escalation


Scenario walkthroughs validated successfully!


### 15. Final Engine Validation & Save

Validate and save the upgraded negotiation engine.

In [41]:
import importlib
import inspect

# Verify all required features
required_features = [
    "def _priority(",
    "def _evaluate_proposal(",
    "def _deadline_check(",
    "def run(",
    "time.perf_counter()",
    'status="AGREED"',
    'status="ESCALATED"',
    'status="FAILED"'
]

for feature in required_features:
    assert feature in negotiation_source, (
        f"Missing feature: {feature}"
    )

# Validate Python syntax
compile(
    negotiation_source,
    str(negotiation_path),
    "exec"
)

# Save final code locally
negotiation_path.write_text(
    negotiation_source,
    encoding="utf-8"
)

# Reload the updated engine
import core.negotiation as negotiation_module
importlib.reload(negotiation_module)

# Reload runner for Notebook 05 compatibility
import core.runner as runner_module
importlib.reload(runner_module)

# Verify integration
final_engine = runner_module.build_engine()

assert isinstance(
    final_engine,
    negotiation_module.NegotiationEngine
)

assert len(final_engine.agents) == 4
assert final_engine.max_rounds == 3

print("All required negotiation features: PASS")
print("Python syntax: PASS")
print("Agent integration: PASS")
print("Notebook 05 interface compatibility: PASS")
print("Final negotiation.py saved successfully!")
print("File:", negotiation_path)

All required negotiation features: PASS
Python syntax: PASS
Agent integration: PASS
Notebook 05 interface compatibility: PASS
Final negotiation.py saved successfully!
File: /content/AgenticX/core/negotiation.py


### 16. Export Final Negotiation Engine

Download the updated negotiation module for GitHub.

In [42]:
from google.colab import files

assert negotiation_path.exists()

assert negotiation_path.read_text(
    encoding="utf-8"
) == negotiation_source

print("Final negotiation.py verified!")
files.download(str(negotiation_path))

Final negotiation.py verified!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>